In [8]:
import rasterio
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import rasterio

from pathlib import Path
from typing import List, Tuple, Optional, Union
from rasterio.enums import MergeAlg
from rasterio import features

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.conversions import create_geodataframe_from_points, create_geodataframe_from_polygons, _rasterize_vector_process, create_binary_raster
from beak.utilities.io import save_raster, check_path, load_raster, load_dataset


# Load data

**User definitions**

In [9]:
# Set base paths and files
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 50
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "mvt_southmid_cont"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")

# Points file and query to select relevant mineral occurences
PATH_LABELS_TRAIN = BASE_PATH / "RAW" / "mineral_deposits" / "Mississippi_Valley_Type" / "TA2_McCafferty_CEUS_splits_from_hack_9m" / "train.csv"
PATH_LABELS_TEST = BASE_PATH / "RAW" / "mineral_deposits" / "Mississippi_Valley_Type" / "TA2_McCafferty_CEUS_splits_from_hack_9m" / "test.csv"
PATH_LABELS_VALID = BASE_PATH / "RAW" / "mineral_deposits" / "Mississippi_Valley_Type" / "TA2_McCafferty_CEUS_splits_from_hack_9m" / "valid.csv"

# Set the output file
PATH_ROOT = BASE_PATH / "PROCESSED" / str("regional" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "labels" / str("MCCAFFERTY_CEUS_TRAIN_TEST_VALID_HM9" + ".tif")

base_raster = rasterio.open(BASE_RASTER)

print(f"Output file: {PATH_EXPORT}")

Output file: S:\Projekte\20230082_DARPA_CriticalMAAS_TA3\Bearbeitung\GitHub\beak-ta3\src\beak\data\PROCESSED\regional_102008_50_mvt_southmid_cont\labels\EPSG_102008_RES_50_mvt_southmid_cont_MCCAFFERTY_CEUS_TRAIN_TEST_VALID_HM9.tif


**Check** data

In [10]:
mineral_sites_train = load_dataset(PATH_LABELS_TRAIN)
mineral_sites_train.drop(columns = ["x", "y", "source"], inplace=True)
mineral_sites_train["source"] = "train"

mineral_sites_train

,label,lon,lat,source
0,1,100750,-355750,train
1,1,98250,-355750,train
2,1,103250,-352250,train
3,1,105250,-350250,train
4,1,101750,-356750,train
...,...,...,...,...
3755,1,1063750,-388750,train
3756,1,1063750,-385750,train
3757,1,99250,-355750,train
3758,1,441750,318250,train


In [11]:
mineral_sites_test = load_dataset(PATH_LABELS_TEST)
mineral_sites_test["source"] = "test"

mineral_sites_test

,label,lon,lat,source
0,1,431750,308250,test
1,1,98750,-348750,test
2,1,1056750,-391750,test
3,1,430750,300250,test
4,1,97750,-356250,test
5,1,99250,-354750,test
6,1,95750,-357750,test
7,1,99250,-356250,test
8,1,103750,-359250,test
9,1,94750,-359250,test


In [12]:
mineral_sites_valid = load_dataset(PATH_LABELS_VALID)
mineral_sites_valid.drop(columns = ["x", "y", "source"], inplace=True)
mineral_sites_valid["source"] = "valid"

mineral_sites_valid


,label,lon,lat,source
0,1,455250,-220750,valid
1,1,95750,-354250,valid
2,1,408250,-287750,valid
3,1,93250,-355250,valid
4,1,276750,-428750,valid
5,1,95250,-355750,valid
6,1,1039750,-349750,valid
7,1,1406250,-24750,valid
8,1,97250,-357250,valid
9,1,98250,-355250,valid


# Create Labels

In [13]:
data = pd.concat([mineral_sites_train, mineral_sites_test, mineral_sites_valid])

gdf = create_geodataframe_from_points(data, long_col="lon", lat_col="lat", crs=base_raster.meta["crs"])
gdf.head()


,label,lon,lat,source,geometry
0,1,100750,-355750,train,POINT (100750.000 -355750.000)
1,1,98250,-355750,train,POINT (98250.000 -355750.000)
2,1,103250,-352250,train,POINT (103250.000 -352250.000)
3,1,105250,-350250,train,POINT (105250.000 -350250.000)
4,1,101750,-356750,train,POINT (101750.000 -356750.000)


In [14]:
labels_array = create_binary_raster(gdf, base_raster, all_touched=False, same_shape=True, fill_negatives=True, out_file=None)
print(f"Number of positive training labels: {np.sum(labels_array==1)}")

Number of positive training labels: 34
